In [1]:
import pandas as pd
import numpy as np
import os
from sdv.single_table import CTGANSynthesizer, GaussianCopulaSynthesizer, TVAESynthesizer
from sdv.metadata import Metadata

In [2]:
# Constants
SYNTHETIC_SAMPLES = 10000
DATA_PATH = "../../../Datasets/csv/spoofing-merged-gps-only.csv"
METADATA_FILE = "spoofing_metadata.json"

In [3]:
# Load the dataset
try:
    df = pd.read_csv(DATA_PATH)
except FileNotFoundError:
    print(f"Error: The file '{DATA_PATH}' was not found.")
    print("Please make sure the path is correct.")
    exit()


In [4]:
# Data Preprocessing
if 'timestamp' in df.columns:
    df = df.drop(columns=['timestamp'])

# Clean up column names
df.columns = df.columns.str.replace('[\\[\\]<>]', '_', regex=True)

# Encode 'label'
if 'label' in df.columns and (df['label'].dtype == object or df['label'].dtype.name == 'category'):
    from sklearn.preprocessing import LabelEncoder

    le = LabelEncoder()
    df['label'] = le.fit_transform(df['label'])

In [5]:
# Load or create metadata
if os.path.exists(METADATA_FILE):
    metadata = Metadata.load_from_json(METADATA_FILE)
    print(f"Loaded metadata from {METADATA_FILE}")
else:
    metadata = Metadata()
    metadata.detect_table_from_dataframe(
        data=df,
        table_name='spoofing_data'
    )
    metadata.save_to_json(METADATA_FILE)
    print(f"Created and saved new metadata to {METADATA_FILE}")

Loaded metadata from spoofing_metadata.json


In [6]:
# Synthesizer configurations
# If you don't have a GPU available, it will automatically use CPU.
synthesizers = [
    {
        # Uses GAN-based, deep learning methods to train a model and generate synthetic data.
        'name': 'CTGAN',
        'class': CTGANSynthesizer,
        'params': {
            'enforce_min_max_values': True,
            'enforce_rounding': False,
            'epochs': 2000,
            'verbose': True,
            'cuda': True
        },
        'output': "../../../Datasets/csv/spoofing-CTGAN-synthetic.csv"
    },
    {
        # Uses classic, statistical methods to train a model and generate synthetic data
        'name': 'GaussianCopula',
        'class': GaussianCopulaSynthesizer,
        'params': {
            'enforce_min_max_values': True,
            'enforce_rounding': False,
        },
        'output': "../../../Datasets/csv/spoofing-GaussianCopula-synthetic.csv"
    },
    {
        # Uses a variational autoencoder (VAE)-based, neural network techniques to train a model and generate synthetic data.
        'name': 'TVAE',
        'class': TVAESynthesizer,
        'params': {
            'enforce_min_max_values': True,
            'enforce_rounding': False,
            'epochs': 2000,
            'verbose': True,
            'cuda': True
        },
        'output': "../../../Datasets/csv/spoofing-TVAE-synthetic.csv"
    }
]

In [7]:
# Fit synthesizers and generate synthetic data
# This will take some time.
for synth in synthesizers:
    print(f"\nProcessing {synth['name']}...")
    SynthClass = synth['class']
    if synth['name'] == 'GaussianCopula':
        synthesizer = SynthClass(metadata)
    else:
        synthesizer = SynthClass(metadata, **synth['params'])
    synthesizer.fit(df)
    synthetic_data = synthesizer.sample(num_rows=SYNTHETIC_SAMPLES)
    print(f"Synthetic data preview for {synth['name']}:")
    print(synthetic_data.head())
    synthetic_data.to_csv(synth['output'], index=False)
    print(f"Synthetic dataset saved to {synth['output']}")


Processing CTGAN...


Gen. (2.70) | Discrim. (0.72): 100%|██████████| 2000/2000 [29:31<00:00,  1.13it/s]  


Synthetic data preview for CTGAN:
       q_0_      q_1_      q_2_      q_3_  delta_q_reset_0_  delta_q_reset_1_  \
0 -0.362032  0.010733  0.011949  0.915282          0.999995     -4.610000e-10   
1 -0.360962  0.036357  0.011443  0.918226          0.999995     -4.610000e-10   
2 -0.153523  0.161991 -0.979830 -0.229867          0.999995     -4.610000e-10   
3 -0.245594  0.166028 -0.957030 -0.112883          0.999995     -4.610000e-10   
4 -0.364505  0.017068  0.004665  0.914580          0.999995     -4.610000e-10   

   delta_q_reset_2_  delta_q_reset_3_  quat_reset_counter      lat_x  ...  \
0      1.580000e-09          0.003237                   3  36.204811  ...   
1      1.580000e-09          0.003237                   3  36.204953  ...   
2      1.580000e-09          0.003237                   3  36.204953  ...   
3      1.580000e-09          0.003237                   3  36.204953  ...   
4      1.580000e-09          0.003237                   3  36.204953  ...   

   v_z_valid  xy

Loss: -233.291: 100%|██████████| 2000/2000 [13:01<00:00,  2.56it/s]


Synthetic data preview for TVAE:
       q_0_      q_1_      q_2_      q_3_  delta_q_reset_0_  delta_q_reset_1_  \
0 -0.379284 -0.012997 -0.009046  0.926133          0.999995     -4.610000e-10   
1 -0.374553  0.002728  0.007273  0.928560          0.999995     -4.610000e-10   
2 -0.373826 -0.015467  0.033817  0.926085          0.999995     -4.610000e-10   
3 -0.370949 -0.002906 -0.011661  0.929975          0.999995     -4.610000e-10   
4 -0.191254  0.091072 -0.979830 -0.071045          0.999995     -4.610000e-10   

   delta_q_reset_2_  delta_q_reset_3_  quat_reset_counter      lat_x  ...  \
0      1.580000e-09          0.003237                   3  36.204847  ...   
1      1.580000e-09          0.003237                   3  36.204811  ...   
2      1.580000e-09          0.003237                   3  36.204811  ...   
3      1.580000e-09          0.003237                   3  36.204811  ...   
4      1.580000e-09          0.003237                   3  36.204811  ...   

   v_z_valid  xy_